# Verify AI Agent Memory & Checkpointer

This notebook demonstrates how to integrate memory checkpointers with the LangGraph ReAct agent, enabling multi-turn conversations and state retention.

### Important Concepts for Developers:

1. **Django Initialization (`setup.py`)**: As always, initializing Django via `setup.init()` is required to load environment variables, database configuration, and the Django settings registry.

2. **Checkpointers**: We use `MemorySaver` (or another checkpointer) to persist the conversational state using `thread_id` from the `RunnableConfig`. This allows the agent to recall past messages and context across chained invocations.

3. **State Propagation (`RunnableConfig`)**: The `config` dictionary is passed into `agent.invoke()`. The `thread_id` acts as the session boundary for memory, and the `user_id` enforces authorization boundaries to mitigate IDOR vulnerabilities inside the tools.

In [ ]:
import setup
setup.init()

In [ ]:
from ai.agents import get_document_agent

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver 

checkpointer = InMemorySaver()

In [ ]:
agent = get_document_agent(model="gemini-flash-latest", checkpointer=checkpointer)

In [ ]:
import uuid 
config = {"configurable": {"user_id": "2", "thread_id": uuid.uuid4()}}

response = agent.invoke(
    {"messages": 
        [
            {"role": "user", "content": "what are my recent documents?"}
        ]
    },
    config
)

In [ ]:
response['messages'][-1].content[0]['text']

In [ ]:
response = agent.invoke(
    {"messages": 
        [
            {"role": "user", "content": "Are there duplicates"}
        ]
    },
    config
)
response['messages'][-1].content[0]['text']